# Evaluate the Akkadian lemmatizer (accuracy and precision) - Colab version

Runs `lemma_model` on `test.conllu` and reports accuracy, precision, recall and F1. For each test token the model sees only the transliterated form and generates `<POS> ... <LEMMA> ... <GLOSS> ...`; that is parsed and compared with the gold columns.

**Before running:** Runtime > Change runtime type > **T4 GPU** (or A100/L4). Put this notebook, `test.conllu` and the `lemma_model` folder in `MyDrive/OB PNs/`. Then run the cells top to bottom.

Results are shown twice: **token-level** (every row counts, so frequent words weigh more) and **type-level** (each distinct form/label combination counts once). The **ceiling** line is the best any model could do from the form alone, since identical forms with different gold labels (like `ša` as REL or DET) can't all be right.

In [ ]:
!pip install -q polars
import os
from collections import Counter

import polars as pl
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = "/content/drive/MyDrive/OB PNs"
os.chdir(WORK_DIR)
print("Working in:", os.getcwd())
print(os.listdir("."))

Mounted at /content/drive
Working in: /content/drive/MyDrive/OB PNs
['test.conllu', 'train.conllu', 'lemma_model', 'lemmatizer_train.ipynb', 'evaluate_lemmatizer.ipynb']


In [ ]:
# ---------------- CONFIG ----------------
MODEL_DIR = "lemma_model"            # trained model folder (in OB PNs)
DATA_FILE = "test.conllu"            # test file (in OB PNs)
OUTPUT_DIR = "."                     # eval_predictions.csv (every distinct row with its prediction) is written here

# Column layout, 1-indexed; negative counts from the end. Same as training.
FORM_COL = 2
LEMMA_COL = 3
POS_COL = 5
GLOSS_COL = 11
KEEP_UNLEMMATIZED = False            # False = skip rows whose lemma is "_" (broken signs "x", numbers), as in training

BATCH_SIZE = 256
NUM_BEAMS = 4
MAX_LENGTH = 64
LIMIT = None                         # e.g. 500 for a quick smoke test; None = whole file
# ----------------------------------------

# tolerate the "connlu" spelling of the filename
if not os.path.exists(DATA_FILE) and os.path.exists("test.connlu"):
    DATA_FILE = "test.connlu"
print("Test file:", DATA_FILE, "| exists:", os.path.exists(DATA_FILE))
print("Model dir:", MODEL_DIR, "| exists:", os.path.isdir(MODEL_DIR))
if os.path.isdir(MODEL_DIR):
    print(os.listdir(MODEL_DIR))

Test file: test.conllu | exists: True
Model dir: lemma_model | exists: True
['checkpoint-24000', 'checkpoint-24543', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


In [ ]:
POS_TAG = "<POS>"
LEMMA_TAG = "<LEMMA>"
GLOSS_TAG = "<GLOSS>"
MISSING = "<MISSING>"

## Evaluation code

In [ ]:
# Data
def _resolve_column(names, index):
    """Resolve a 1-indexed (possibly negative) column position to its column name."""
    n = len(names)
    pos = index if index > 0 else n + index + 1
    if pos < 1 or pos > n:
        raise ValueError(f"Column index {index} is out of range for a file with {n} columns.")
    return names[pos - 1]


def load_gold(path, form_col, lemma_col, pos_col, gloss_col, drop_unlemmatized=True):
    """Return a DataFrame with columns form, pos, lemma, gloss (all strings)."""
    lazy = pl.scan_csv(
        path,
        has_header=False,
        separator="\t",
        quote_char=None,
        comment_prefix="#",
        infer_schema_length=0,  # read everything as strings
    )
    names = lazy.collect_schema().names()
    print(f"Detected {len(names)} columns in {path}.")
    lazy = lazy.select(
        form=pl.col(_resolve_column(names, form_col)),
        pos=pl.col(_resolve_column(names, pos_col)),
        lemma=pl.col(_resolve_column(names, lemma_col)),
        gloss=pl.col(_resolve_column(names, gloss_col)),
    ).drop_nulls()
    if drop_unlemmatized:
        lazy = lazy.filter(pl.col("lemma") != "_")
    df = lazy.collect()
    if df.is_empty():
        raise ValueError("No rows loaded. Check the column settings against this file's layout.")
    return df


def parse_prediction(text):
    """Turn '<POS> N <LEMMA> māru <GLOSS> son' back into {pos, lemma, gloss}."""
    result = {"pos": None, "lemma": None, "gloss": None}
    tags = (POS_TAG, LEMMA_TAG, GLOSS_TAG)
    for tag, key in zip(tags, ("pos", "lemma", "gloss")):
        if tag in text:
            start = text.index(tag) + len(tag)
            end = len(text)
            for other in tags:
                if other != tag:
                    idx = text.find(other, start)
                    if idx != -1:
                        end = min(end, idx)
            result[key] = text[start:end].strip()
    return result

In [ ]:
# Model inference
def predict_forms(forms, model_dir, batch_size, num_beams, max_length):
    """Generate one output string per distinct form. Returns {form: text}."""
    import torch  # imported here so the metrics code can be used without torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model from {model_dir} on {device} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device).eval()

    order = sorted(forms, key=len)  # similar lengths per batch = less padding
    out = {}
    for i in range(0, len(order), batch_size):
        batch = order[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            # no_repeat_ngram_size=0: bart-base's default config blocks repeated
            # 3-grams, which would corrupt names whose lemma and gloss are the
            # same word (e.g. "Idrimi ... Idrimi").
            gen = model.generate(**enc, num_beams=num_beams, max_length=max_length, no_repeat_ngram_size=0)
        texts = tokenizer.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        out.update(zip(batch, texts))
        done = min(i + batch_size, len(order))
        if (i // batch_size) % 10 == 0 or done == len(order):
            print(f"  predicted {done:,}/{len(order):,} distinct forms")
    return out

In [ ]:
# Metrics
def field_metrics(gold, pred):
    """Accuracy and precision for a free-text field (lemma or gloss)."""
    n = len(gold)
    correct = sum(g == p for g, p in zip(gold, pred))
    nonempty = sum(1 for p in pred if p)
    return {
        "accuracy": correct / n,
        "precision": correct / nonempty if nonempty else 0.0,
        "n_correct": correct,
        "n_predicted": nonempty,
    }


def pos_metrics(gold, pred):
    """Accuracy plus per-class and macro / weighted precision, recall, F1 for POS."""
    pred = [p if p else MISSING for p in pred]
    labels = sorted(set(gold) | set(pred))
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    per_class = {}
    for label in labels:
        prec = tp[label] / (tp[label] + fp[label]) if tp[label] + fp[label] else 0.0
        rec = tp[label] / (tp[label] + fn[label]) if tp[label] + fn[label] else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        per_class[label] = {"precision": prec, "recall": rec, "f1": f1, "support": tp[label] + fn[label]}
    total = len(gold)
    k = len(per_class)
    return {
        "accuracy": sum(tp.values()) / total,
        "macro_precision": sum(v["precision"] for v in per_class.values()) / k,
        "macro_recall": sum(v["recall"] for v in per_class.values()) / k,
        "macro_f1": sum(v["f1"] for v in per_class.values()) / k,
        "weighted_precision": sum(v["precision"] * v["support"] for v in per_class.values()) / total,
        "weighted_recall": sum(v["recall"] * v["support"] for v in per_class.values()) / total,
        "weighted_f1": sum(v["f1"] * v["support"] for v in per_class.values()) / total,
        "per_class": per_class,
    }


def evaluate(gold, pred):
    """gold / pred: lists of dicts with keys pos, lemma, gloss (pred values may be None)."""
    n = len(gold)
    g = {k: [r[k] for r in gold] for k in ("pos", "lemma", "gloss")}
    p = {k: [r[k] for r in pred] for k in ("pos", "lemma", "gloss")}
    all_ok = sum(all(gr[k] == pr[k] for k in ("pos", "lemma", "gloss")) for gr, pr in zip(gold, pred))
    malformed = sum(any(not pr[k] for k in ("pos", "lemma", "gloss")) for pr in pred)
    return {
        "n": n,
        "full_match_accuracy": all_ok / n,
        "malformed_outputs": malformed,
        "pos": pos_metrics(g["pos"], p["pos"]),
        "lemma": field_metrics(g["lemma"], p["lemma"]),
        "gloss": field_metrics(g["gloss"], p["gloss"]),
    }


def ceiling(gold_rows):
    """Best possible full-match accuracy if each form gets one fixed answer."""
    by_form = {}
    for form, r in gold_rows:
        by_form.setdefault(form, Counter())[(r["pos"], r["lemma"], r["gloss"])] += 1
    best = sum(c.most_common(1)[0][1] for c in by_form.values())
    return best / len(gold_rows)

In [ ]:
# Reporting
def pct(x):
    return f"{100 * x:6.2f}%"


def print_report(token, typ, ceilings, n_forms):
    print("\n" + "=" * 66)
    print(f"{'':32}{'token-level':>16}{'type-level':>16}")
    print(f"{'rows scored':32}{token['n']:>16,}{typ['n']:>16,}")
    print("-" * 66)
    rows = [
        ("full match (all 3 fields)", "full_match_accuracy", None),
        ("POS accuracy", "accuracy", "pos"),
        ("POS macro precision", "macro_precision", "pos"),
        ("POS macro recall", "macro_recall", "pos"),
        ("POS macro F1", "macro_f1", "pos"),
        ("POS weighted precision", "weighted_precision", "pos"),
        ("lemma accuracy", "accuracy", "lemma"),
        ("lemma precision", "precision", "lemma"),
        ("gloss accuracy", "accuracy", "gloss"),
        ("gloss precision", "precision", "gloss"),
    ]
    for label, key, field in rows:
        a = token[key] if field is None else token[field][key]
        b = typ[key] if field is None else typ[field][key]
        print(f"{label:32}{pct(a):>16}{pct(b):>16}")
    print("-" * 66)
    print(f"{'malformed outputs':32}{token['malformed_outputs']:>16,}{typ['malformed_outputs']:>16,}")
    print(f"{'full-match ceiling (form only)':32}{pct(ceilings['token']):>16}{pct(ceilings['type']):>16}")
    print("=" * 66)
    print(f"Distinct forms predicted: {n_forms:,}")

    print("\nPOS per-class results (token-level, all classes, most frequent first):")
    print(f"{'class':12}{'precision':>11}{'recall':>9}{'f1':>9}{'support':>10}")
    classes = sorted(token["pos"]["per_class"].items(), key=lambda kv: -kv[1]["support"])
    for label, v in classes:
        print(f"{label:12}{pct(v['precision']):>11}{pct(v['recall']):>9}{pct(v['f1']):>9}{v['support']:>10,}")


def print_top_errors(rows, field, k=10):
    errors = Counter()
    for r in rows:
        if r[f"gold_{field}"] != r[f"pred_{field}"]:
            errors[(r["form"], r[f"gold_{field}"], r[f"pred_{field}"])] += r["count"]
    print(f"\nMost frequent {field} errors (form | gold -> predicted | count):")
    if not errors:
        print("  none")
    for (form, gold, pred), c in errors.most_common(k):
        print(f"  {form} | {gold} -> {pred} | {c:,}")

In [ ]:
# Scoring and output files
def run(df, pred_map, output_dir):
    """Score df (form/pos/lemma/gloss) against pred_map {form: generated text}."""
    parsed = {form: parse_prediction(text) for form, text in pred_map.items()}

    # token level: every row counts
    forms = df["form"].to_list()
    gold_tok = [{"pos": a, "lemma": b, "gloss": c} for a, b, c in zip(df["pos"], df["lemma"], df["gloss"])]
    pred_tok = [parsed[f] for f in forms]

    # type level: each distinct (form, pos, lemma, gloss) once, with its frequency
    counts = Counter(zip(forms, df["pos"].to_list(), df["lemma"].to_list(), df["gloss"].to_list()))
    uniq = list(counts)
    gold_typ = [{"pos": p, "lemma": l, "gloss": g} for _, p, l, g in uniq]
    pred_typ = [parsed[f] for f, _, _, _ in uniq]

    token = evaluate(gold_tok, pred_tok)
    typ = evaluate(gold_typ, pred_typ)
    ceilings = {
        "token": ceiling(list(zip(forms, gold_tok))),
        "type": ceiling([(f, r) for (f, _, _, _), r in zip(uniq, gold_typ)]),
    }

    rows = []
    for (form, p, l, g), pr in zip(uniq, pred_typ):
        rows.append(
            {
                "form": form,
                "count": counts[(form, p, l, g)],
                "gold_pos": p,
                "gold_lemma": l,
                "gold_gloss": g,
                "pred_pos": pr["pos"] or "",
                "pred_lemma": pr["lemma"] or "",
                "pred_gloss": pr["gloss"] or "",
                "pos_ok": pr["pos"] == p,
                "lemma_ok": pr["lemma"] == l,
                "gloss_ok": pr["gloss"] == g,
                "all_ok": pr["pos"] == p and pr["lemma"] == l and pr["gloss"] == g,
            }
        )
    rows.sort(key=lambda r: -r["count"])

    print_report(token, typ, ceilings, len(pred_map))
    print_top_errors(rows, "lemma")
    print_top_errors(rows, "pos")
    print_top_errors(rows, "gloss")

    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, "eval_predictions.csv")
    pl.DataFrame(rows).write_csv(csv_path)
    print(f"\nWrote {csv_path}")
    return token, typ

## Run
Loads the test file, predicts each distinct form once, then scores every row.

In [ ]:
df = load_gold(DATA_FILE, FORM_COL, LEMMA_COL, POS_COL, GLOSS_COL, drop_unlemmatized=not KEEP_UNLEMMATIZED)
if LIMIT:
    df = df.head(LIMIT)
distinct = df["form"].unique().to_list()
print(f"Scoring {len(df):,} rows ({len(distinct):,} distinct forms).")

with pl.Config(fmt_str_lengths=60):
    print(df.head(8))

pred_map = predict_forms(distinct, MODEL_DIR, BATCH_SIZE, NUM_BEAMS, MAX_LENGTH)

Detected 17 columns in test.conllu.
Scoring 73,866 rows (17,633 distinct forms).
shape: (8, 4)
┌────────┬─────┬────────┬────────────┐
│ form   ┆ pos ┆ lemma  ┆ gloss      │
│ ---    ┆ --- ┆ ---    ┆ ---        │
│ str    ┆ str ┆ str    ┆ str        │
╞════════╪═════╪════════╪════════════╡
│ MU-šu₂ ┆ N   ┆ šumu   ┆ name       │
│ ŠU₂    ┆ V   ┆ erēpu  ┆ cloud over │
│ AN     ┆ N   ┆ šamû   ┆ sky        │
│ UTAH   ┆ V   ┆ natāku ┆ drip       │
│ SI     ┆ N   ┆ ubānu  ┆ finger     │
│ ana    ┆ PRP ┆ ana    ┆ to         │
│ ŠU₂    ┆ N   ┆ erbu   ┆ (sun)set   │
│ LAL    ┆ V   ┆ nahāsu ┆ (re)cede   │
└────────┴─────┴────────┴────────────┘
Loading model from lemma_model on cuda ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  predicted 256/17,633 distinct forms
  predicted 2,816/17,633 distinct forms
  predicted 5,376/17,633 distinct forms
  predicted 7,936/17,633 distinct forms
  predicted 10,496/17,633 distinct forms
  predicted 13,056/17,633 distinct forms
  predicted 15,616/17,633 distinct forms
  predicted 17,633/17,633 distinct forms


In [ ]:
token, typ = run(df, pred_map, OUTPUT_DIR)


                                     token-level      type-level
rows scored                               73,866          18,888
------------------------------------------------------------------
full match (all 3 fields)                 83.34%          68.73%
POS accuracy                              91.98%          87.58%
POS macro precision                       64.93%          59.69%
POS macro recall                          61.54%          55.54%
POS macro F1                              62.37%          56.39%
POS weighted precision                    92.07%          88.16%
lemma accuracy                            90.66%          80.03%
lemma precision                           90.66%          80.03%
gloss accuracy                            84.51%          70.95%
gloss precision                           84.51%          70.95%
------------------------------------------------------------------
malformed outputs                              2               1
full-match ceiling (

## Browse the mistakes
Most frequent wrong predictions first (also saved in `eval_predictions.csv`).

In [ ]:
res = pl.read_csv(os.path.join(OUTPUT_DIR, "eval_predictions.csv"))
with pl.Config(tbl_rows=40, fmt_str_lengths=40, tbl_cols=-1):
    print(res.filter(~pl.col("all_ok")).head(40))

shape: (40, 12)
┌────────┬───────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ form   ┆ count ┆ gold_p ┆ gold_l ┆ gold_ ┆ pred_ ┆ pred_ ┆ pred_ ┆ pos_o ┆ lemma ┆ gloss ┆ all_o │
│ ---    ┆ ---   ┆ os     ┆ emma   ┆ gloss ┆ pos   ┆ lemma ┆ gloss ┆ k     ┆ _ok   ┆ _ok   ┆ k     │
│ str    ┆ i64   ┆ ---    ┆ ---    ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│        ┆       ┆ str    ┆ str    ┆ str   ┆ str   ┆ str   ┆ str   ┆ bool  ┆ bool  ┆ bool  ┆ bool  │
╞════════╪═══════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ ša₂    ┆ 821   ┆ REL    ┆ ša     ┆ that  ┆ DET   ┆ ša    ┆ of    ┆ false ┆ true  ┆ false ┆ false │
│ ša     ┆ 408   ┆ DET    ┆ ša     ┆ of    ┆ REL   ┆ ša    ┆ that  ┆ false ┆ true  ┆ false ┆ false │
│ ME     ┆ 158   ┆ NU     ┆ meʾatu ┆ (one) ┆ N     ┆ ūmu   ┆ day   ┆ false ┆ false ┆ false ┆ false │
│        ┆       ┆        ┆        ┆ hundr ┆       ┆       ┆       ┆       